# camel beauty assessor — experiments

mosaic is off for every run here. it stitches four photos into one and halves every object, which
is the wrong direction for classes already at 19 to 32 pixels. `e0-nomosaic` measures that on its
own, and the rest build on it one change at a time.

| run | compared against | isolates |
|---|---|---|
| `e0-nomosaic` | `b1-rgb` | mosaic |
| `e1-gray` | `e0-nomosaic` | colour |
| `e2-rot45` | `e0-nomosaic` | 45 degree turns |
| `e3-rot45-gray` | `e2-rot45` | colour, with the turns in place |
| `e4-scratch` | `e0-nomosaic` | pretraining |
| `e5-hires` | `e0-nomosaic` | image size, 640 against 1280 |

same split and same test images as `baseline.ipynb`, so every row here is comparable to every row
there.

## 1. setup

### 1.1 imports

In [ ]:
import os
import csv
import json
import math
import time
import shutil
import pathlib
import platform
import collections

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import mlflow
from ultralytics import YOLO, settings as yolo_settings

print(f"torch {torch.__version__}   cuda {torch.cuda.is_available()}")

### 1.2 paths

In [ ]:
root = pathlib.Path.cwd()

VERSION = "v2"
data = root / "dataset" / VERSION

src_rgb = data / "split" / "rgb"
src_gray = data / "split" / "gray"

rot_rgb = data / "augmented" / "rot45-rgb"
rot_gray = data / "augmented" / "rot45-gray"

det_rgb = data / "detect" / "plain"
det_gray = data / "detect" / "gray"
det_rot = data / "detect" / "rot45-rgb"
det_rot_gray = data / "detect" / "rot45-gray"

project = root / "runs" / VERSION
results = root / "results" / VERSION
weights_dir = root / "weights"

### 1.3 display options

In [ ]:
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 115)
pd.set_option("display.expand_frame_repr", False)

### 1.4 run configuration

In [ ]:
seed = 42
splits = ["train", "val", "test"]
rotations = [0, 45, 135, 225, 315]

canon = ["Camel", "High_withers", "Large-head", "Large-lips", "Large-nose",
         "Large_hump", "Long-legs", "Long-neck", "Wide_body"]
traits = [c for c in canon if c != "Camel"]

epochs = 200
imgsz = 640
imgsz_hi = 1280
patience = 50
conf_threshold = 0.25
iou_threshold = 0.5
mosaic = 0.0

### 1.5 device

In [ ]:
device = 0 if torch.cuda.is_available() else "cpu"
batch = 16 if device != "cpu" else 8
workers = 8 if device != "cpu" else 2
amp = device != "cpu"

print(f"device {device}   batch {batch}   workers {workers}   amp {amp}")
print(f"cuda: {torch.cuda.get_device_name(0)}")


### 1.6 experiment tracking

In [ ]:
tracking_uri = "sqlite:///" + (root / "mlflow.db").resolve().as_posix()
os.environ["MLFLOW_TRACKING_URI"] = tracking_uri
os.environ["MLFLOW_EXPERIMENT_NAME"] = f"camel-beauty-cv-{VERSION}"
yolo_settings.update({"mlflow": True})

mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment(f"camel-beauty-cv-{VERSION}")

print("tracking to", tracking_uri)

## 2. helpers

### 2.1 labels

In [ ]:
def cls_name(i):
    i = int(i)
    if not 0 <= i < len(canon):
        raise ValueError(f"class id {i} is outside the {len(canon)} class schema {canon}")
    return canon[i]


def read_label(path):
    rows = []
    with open(path, encoding="utf-8") as fh:
        for n, line in enumerate(fh, 1):
            if not line.strip():
                continue
            p = line.split()
            if not 0 <= int(p[0]) < len(canon):
                raise ValueError(f"{path} line {n}: class id {p[0]} is not one of the {len(canon)}")
            rows.append((int(p[0]), [float(v) for v in p[1:]]))
    return rows


def label_for(root, sp, name):
    return root / sp / "labels" / (os.path.splitext(name)[0] + ".txt")


def to_box(geo):
    if len(geo) == 4:
        xc, yc, w, h = geo
    else:
        xs, ys = geo[0::2], geo[1::2]
        x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
        xc, yc, w, h = (x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0
    return [min(max(v, 0.0), 1.0) for v in (xc, yc, w, h)]

### 2.2 geometry

In [ ]:
def xywhn_to_xyxyn(box):
    xc, yc, w, h = box
    return [xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2]


def overlap(a, b):
    x0, y0 = max(a[0], b[0]), max(a[1], b[1])
    x1, y1 = min(a[2], b[2]), min(a[3], b[3])
    return max(0.0, x1 - x0) * max(0.0, y1 - y0)


def iou(a, b):
    inter = overlap(a, b)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def place(src, dst):
    try:
        os.link(src, dst)
    except (OSError, NotImplementedError, AttributeError):
        shutil.copy2(src, dst)

## 3. rotation

a 90 degree turn is exact. a 45 degree turn is not: pixels are resampled, the canvas grows, and an
upright box becomes a diamond whose enclosing box is larger than the original. polygon rows keep
their true shape, plain box rows get looser.

In [ ]:
def rotate_points(pts, deg, w, h):
    a = math.radians(deg)
    ca, sa = math.cos(a), math.sin(a)
    nw, nh = abs(w * ca) + abs(h * sa), abs(w * sa) + abs(h * ca)
    return [((dx * ca + dy * sa + nw / 2) / nw, (-dx * sa + dy * ca + nh / 2) / nh)
            for dx, dy in ((x * w - w / 2, y * h - h / 2) for x, y in pts)]


def rotate_line(parts, deg, w, h):
    v = [float(x) for x in parts[1:]]
    if len(v) == 4:
        xc, yc, bw, bh = v
        corners = [(xc - bw / 2, yc - bh / 2), (xc + bw / 2, yc - bh / 2),
                   (xc + bw / 2, yc + bh / 2), (xc - bw / 2, yc + bh / 2)]
        xs, ys = zip(*rotate_points(corners, deg, w, h))
        v = [(min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2, max(xs) - min(xs), max(ys) - min(ys)]
    else:
        v = [c for p in rotate_points(list(zip(v[0::2], v[1::2])), deg, w, h) for c in p]
    return parts[0] + " " + " ".join(f"{min(max(c, 0.0), 1.0):.6f}" for c in v)

In [ ]:
def build_rot(source, dst):
    shutil.rmtree(dst, ignore_errors=True)
    made = collections.Counter()

    for sp in splits:
        (dst / sp / "images").mkdir(parents=True)
        (dst / sp / "labels").mkdir(parents=True)

        for name in sorted(os.listdir(source / sp / "images")):
            stem, ext = os.path.splitext(name)
            for deg in (rotations if sp == "train" else [0]):
                tag = stem if deg == 0 else f"{stem}_r{deg}"
                if deg == 0:
                    place(source / sp / "images" / name, dst / sp / "images" / (tag + ext))
                    shutil.copy2(label_for(source, sp, name), dst / sp / "labels" / (tag + ".txt"))
                else:
                    img = Image.open(source / sp / "images" / name)
                    w, h = img.size
                    img.rotate(deg, expand=True, resample=Image.BICUBIC).save(
                        dst / sp / "images" / (tag + ext), quality=95)
                    rows = [p.split() for p in
                            open(label_for(source, sp, name), encoding="utf-8").read().splitlines() if p.strip()]
                    (dst / sp / "labels" / (tag + ".txt")).write_text(
                        "".join(rotate_line(p, deg, w, h) + "\n" for p in rows), encoding="utf-8")
                made[sp] += 1

    shutil.copy2(source / "data.yaml", dst / "data.yaml")
    return made


for source, dst in ((src_rgb, rot_rgb), (src_gray, rot_gray)):
    print(f"{dst.name:<22} {dict(build_rot(source, dst))}")

## 4. label repair

In [ ]:
def build_det(src_root, dst_root):
    assert (src_root / "train" / "images").is_dir(), f"{src_root} is missing"

    shutil.rmtree(dst_root, ignore_errors=True)
    stat = collections.Counter()

    for sp in splits:
        (dst_root / sp / "images").mkdir(parents=True)
        (dst_root / sp / "labels").mkdir(parents=True)

        for name in sorted(os.listdir(src_root / sp / "images")):
            place(src_root / sp / "images" / name, dst_root / sp / "images" / name)
            rows = read_label(label_for(src_root, sp, name))
            stat["files"] += 1
            stat["lines"] += len(rows)
            out = [f"{cls} " + " ".join(f"{v:.6f}" for v in to_box(geo)) for cls, geo in rows]
            label_for(dst_root, sp, name).write_text("\n".join(out) + "\n", encoding="utf-8")

    (dst_root / "data.yaml").write_text(
        f"path: {dst_root.resolve().as_posix()}\n"
        "train: train/images\nval: val/images\ntest: test/images\n\n"
        f"nc: {len(canon)}\n"
        "names: [" + ", ".join(f"'{c}'" for c in canon) + "]\n",
        encoding="utf-8")

    return stat, dst_root / "data.yaml"


stat_rgb, data_rgb = build_det(src_rgb, det_rgb)
stat_gray, data_gray = build_det(src_gray, det_gray)
stat_rot, data_rot = build_det(rot_rgb, det_rot)
stat_rot_gray, data_rot_gray = build_det(rot_gray, det_rot_gray)

pd.DataFrame({"rgb": stat_rgb, "gray": stat_gray,
              "rot45": stat_rot, "rot45 gray": stat_rot_gray}).fillna(0).astype(int)

## 5. verification

In [ ]:
checks = {}
bad = []
for dst in (det_rgb, det_gray, det_rot, det_rot_gray):
    for sp in splits:
        for name in sorted(os.listdir(dst / sp / "images")):
            lp = label_for(dst, sp, name)
            if not lp.exists():
                bad.append(("missing label", lp.name))
                continue
            for cls, geo in read_label(lp):
                if len(geo) != 4 or any(v < 0 or v > 1 for v in geo):
                    bad.append(("bad geometry", lp.name))

checks["every image has a valid five field label"] = not bad
checks[f"rot45 train grew by exactly {len(rotations)}x"] = (
    len(os.listdir(det_rot / "train" / "images")) == len(os.listdir(src_rgb / "train" / "images")) * len(rotations))
checks["val and test were not rotated"] = all(
    len(os.listdir(det_rot / sp / "images")) == len(os.listdir(src_rgb / sp / "images")) for sp in ("val", "test"))
checks["no rotated file reached val or test"] = not any(
    os.path.splitext(f)[0].endswith(tuple(f"_r{d}" for d in rotations if d))
    for sp in ("val", "test") for f in os.listdir(det_rot / sp / "images"))
checks["the two rot45 sets differ in nothing but colour"] = all(
    label_for(det_rot, sp, f).read_text(encoding="utf-8") == label_for(det_rot_gray, sp, f).read_text(encoding="utf-8")
    for sp in splits for f in os.listdir(det_rot / sp / "images"))
checks["gray and rgb share the same test images"] = (
    sorted(os.listdir(det_gray / "test" / "images")) == sorted(os.listdir(det_rot / "test" / "images")))

for k, v in checks.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")
assert all(checks.values()), "a verification check failed"

### 5.1 what the 45 degree turn cost the labels

In [ ]:
def median_area(root, sp):
    a = collections.defaultdict(list)
    for f in os.listdir(root / sp / "labels"):
        for cls, (xc, yc, w, h) in read_label(root / sp / "labels" / f):
            a[cls_name(cls)].append(w * h)
    return pd.Series({k: float(np.median(v)) for k, v in a.items()})


grew = pd.DataFrame({"upright": median_area(det_gray, "train"), "after 45": median_area(det_rot, "train")})
grew["x bigger"] = (grew["after 45"] / grew["upright"]).round(2)
grew.reindex(canon)

## 6. training runs

In [ ]:
def train_run(name, data, init, weights="yolo11n.pt", size=None):
    size = size or imgsz
    os.environ["MLFLOW_RUN"] = name
    started = time.time()
    model = YOLO(str(weights_dir / weights) if weights.endswith(".pt") else weights)
    model.train(data=str(data), project=str(project), name=name, exist_ok=True,
                epochs=epochs, imgsz=size, batch=batch, device=device, workers=workers,
                seed=seed, deterministic=True, amp=amp, patience=patience, plots=True,
                mosaic=mosaic)

    return {"run": name, "init": init, "dataset": pathlib.Path(data).parent.name, "weights": weights,
            "best": str(project / name / "weights" / "best.pt"),
            "epochs": epochs, "imgsz": size, "batch": batch, "device": str(device),
            "minutes": round((time.time() - started) / 60, 1)}


def evaluate(rec, data, split="test"):
    m = YOLO(rec["best"]).val(data=str(data), split=split, imgsz=rec["imgsz"], batch=batch, device=device,
                              project=str(project), name=f"{rec['run']}-{split}", exist_ok=True, plots=True)
    rec = dict(rec)
    rec.update({"split": split, "mAP50": round(float(m.box.map50), 4), "mAP50_95": round(float(m.box.map), 4),
                "precision": round(float(m.box.mp), 4), "recall": round(float(m.box.mr), 4),
                "val_dir": str(m.save_dir)})
    return rec, m


def per_class(m):
    rows = []
    for i, c in enumerate(list(m.box.ap_class_index)):
        p, r, ap50, ap = m.box.class_result(i)
        rows.append({"cls": cls_name(c), "precision": round(float(p), 3), "recall": round(float(r), 3),
                     "mAP50": round(float(ap50), 3), "mAP50_95": round(float(ap), 3)})
    return pd.DataFrame(rows).set_index("cls").reindex(canon)

In [ ]:
e0 = train_run("e0-nomosaic", data_rgb, "coco")
e0

In [ ]:
e1 = train_run("e1-gray", data_gray, "coco")
e1

In [ ]:
e2 = train_run("e2-rot45", data_rot, "coco")
e2

In [ ]:
e3 = train_run("e3-rot45-gray", data_rot_gray, "coco")
e3

In [ ]:
e4 = train_run("e4-scratch", data_rgb, "random", weights="yolo11n.yaml")
e4

In [ ]:
e5 = train_run("e5-hires", data_rgb, "coco", size=imgsz_hi)
e5

## 7. learning curves

In [ ]:
def history(rec):
    path = pathlib.Path(rec["best"]).parents[1] / "results.csv"
    if not path.exists():
        return None
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    return df


def loss_pair(df):
    train = df[[c for c in df.columns if c.startswith("train/") and c.endswith("_loss")]].sum(axis=1)
    val = df[[c for c in df.columns if c.startswith("val/") and c.endswith("_loss")]].sum(axis=1)
    return train, val


def map_col(df):
    return next(c for c in df.columns if "mAP50-95" in c)


trained = [r for r in (e0, e1, e2, e3, e4, e5) if history(r) is not None]
assert trained, "no results.csv found, run section 6 first"

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 8), sharex=True)
for ax, rec in zip(axes.ravel(), trained):
    df = history(rec)
    train, val = loss_pair(df)
    best_epoch = int(df.epoch.iloc[df[map_col(df)].values.argmax()])

    ax.plot(df.epoch, train, label="train", color="#4c72b0")
    ax.plot(df.epoch, val, label="val", color="#c44e52", linestyle="--")
    ax.axvline(best_epoch, color="#55a868", lw=1, label=f"best epoch {best_epoch}")
    ax.set_title(rec["run"], fontsize=10)
    ax.set_xlabel("epoch")
    ax.set_ylabel("box + cls + dfl loss")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for rec in trained:
    df = history(rec)
    train, val = loss_pair(df)
    axes[0].plot(df.epoch, df[map_col(df)], label=rec["run"])
    axes[1].plot(df.epoch, val - train, label=rec["run"])

axes[0].set_title("val mAP50-95 per epoch")
axes[1].set_title("val loss minus train loss: the memorising gap")
axes[1].axhline(0, color="grey", lw=0.8)
for ax in axes:
    ax.set_xlabel("epoch")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
rows = []
for rec in trained:
    df = history(rec)
    train, val = loss_pair(df)
    pos = int(df[map_col(df)].values.argmax())
    rows.append({"run": rec["run"], "epochs_run": int(df.epoch.max()),
                 "best_epoch": int(df.epoch.iloc[pos]),
                 "best_val_mAP50_95": round(float(df[map_col(df)].max()), 4),
                 "train_loss_end": round(float(train.iloc[-1]), 3),
                 "val_loss_best": round(float(val.min()), 3),
                 "val_loss_end": round(float(val.iloc[-1]), 3)})

curves = pd.DataFrame(rows).set_index("run")
curves["gap"] = (curves.val_loss_end - curves.train_loss_end).round(3)
curves["val_rebound"] = (curves.val_loss_end - curves.val_loss_best).round(3)
curves

## 8. results

In [ ]:
e0_test, m0 = evaluate(e0, data_rgb)
e1_test, m1 = evaluate(e1, data_gray)
e2_test, m2 = evaluate(e2, data_rot)
e3_test, m3 = evaluate(e3, data_rot_gray)
e4_test, m4 = evaluate(e4, data_rgb)
e5_test, m5 = evaluate(e5, data_rgb)

runs = [e0_test, e1_test, e2_test, e3_test, e4_test, e5_test]
compare = pd.DataFrame(runs)[
    ["run", "dataset", "imgsz", "minutes", "mAP50", "mAP50_95", "precision", "recall"]].set_index("run")
compare

### 8.1 against the baseline

`experiments.csv` already holds the four baseline runs, so the two notebooks line up in one table
without anything being re-run.

In [ ]:
prev = pd.read_csv(results / "experiments.csv")
prev = prev.drop_duplicates("run", keep="last").set_index("run")[
    ["dataset", "imgsz", "minutes", "mAP50", "mAP50_95", "precision", "recall"]]

everything = pd.concat([prev, compare]).sort_values("mAP50_95", ascending=False)
everything["vs b1"] = (everything.mAP50_95 - prev.loc["b1-rgb", "mAP50_95"]).round(4)
everything

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
everything.mAP50_95.plot(kind="bar", ax=ax, color="#4c72b0")
ax.axhline(prev.loc["b1-rgb", "mAP50_95"], color="#c44e52", lw=1.2, linestyle="--", label="b1-rgb")
ax.set_ylabel("test mAP50-95")
ax.set_xlabel("")
ax.legend()
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

### 8.2 per-class

In [ ]:
pc = pd.concat({"e0-nomosaic": per_class(m0), "e2-rot45": per_class(m2),
                "e5-hires": per_class(m5)}, axis=1)
pc["boxes_in_test"] = [sum(1 for f in os.listdir(det_rot / "test" / "labels")
                           for cls, _ in read_label(det_rot / "test" / "labels" / f)
                           if cls_name(cls) == c) for c in canon]
pc

## 9. mlops artifacts

In [ ]:
for rec in runs:
    with mlflow.start_run(run_name=rec["run"] + "-test"):
        mlflow.log_params({k: rec[k] for k in
                           ("init", "dataset", "weights", "epochs", "imgsz", "batch", "device", "split")})
        mlflow.log_metrics({k: rec[k] for k in ("mAP50", "mAP50_95", "precision", "recall")})
        mlflow.log_metric("train_minutes", rec["minutes"])
        for art in ("confusion_matrix_normalized.png", "PR_curve.png"):
            f = pathlib.Path(rec["val_dir"]) / art
            if f.exists():
                mlflow.log_artifact(str(f))

table = pd.DataFrame(runs)[
    ["run", "init", "dataset", "weights", "epochs", "imgsz", "batch", "device", "minutes",
     "mAP50", "mAP50_95", "precision", "recall", "best"]]
table.insert(0, "logged_at", pd.Timestamp.now().isoformat(timespec="seconds"))
table.to_csv(results / "experiments.csv", mode="a", header=False, index=False)

pc.to_csv(results / "per_class_experiments.csv")
everything.to_csv(results / "all_runs.csv")

print(f"{len(table)} rows appended to {results / 'experiments.csv'}")
pd.read_csv(results / "experiments.csv")